<a href="https://colab.research.google.com/github/gabiuxo/Algoritmos-de-Aprendizaje-Automatico/blob/main/PracticaJuevesTema14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica jueves - Tema 14: Documentación efectiva y gobernanza de modelos

**Gabriel Elizondo Martinez**  
**Matrícula:** AL07009102


## Reto 1. Entrenamiento reproducible con semilla fija y artefacto


In [ ]:
import json
from pathlib import Path

import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

# set the seed and create reproducible data
np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=6, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# train the classifier with fixed parameters
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
f1_val = f1_score(y_test, model.predict(X_test))

# export the initial metrics artifact
report = {"seed": 42, "n_estimators": 50, "test_f1": float(f1_val)}
Path("artefactos").mkdir(exist_ok=True)
Path("artefactos/metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Base metrics exported:", report)


Base metrics exported: {'seed': 42, 'n_estimators': 50, 'test_f1': 0.9}


Fijé la semilla, el conjunto de datos y los parámetros principales para poder repetir el entrenamiento. El archivo `metrics.json` guarda la métrica inicial, aunque una réplica completa también requeriría documentar la versión exacta del dataset y las dependencias del entorno.


## Reto 2. Auditoría de condiciones de reproducibilidad


In [ ]:
# compare two reported F1 scores using the agreed tolerance
test_f1_a = 0.850
test_f1_b = 0.841
epsilon = 0.02
difference = abs(test_f1_a - test_f1_b)

if difference < epsilon:
    result = "réplica aprobada"
else:
    result = "réplica rechazada"

print(f"F1(A): {test_f1_a:.3f}")
print(f"F1(B): {test_f1_b:.3f}")
print(f"Difference: {difference:.3f}")
print(f"Tolerance: {epsilon:.3f}")
print(result)


F1(A): 0.850
F1(B): 0.841
Difference: 0.009
Tolerance: 0.020
réplica aprobada


Comparé los dos resultados usando la regla |F1(A) - F1(B)| < ε. La diferencia fue de 0.009, menor que la tolerancia de 0.02, por lo que aprobé la réplica como equivalente dentro del margen definido.


## Reto 3. Bitácora de decisiones


In [ ]:
decision_log = (
    "| Fecha | Decisión | Evidencia | Responsable | Impacto |\n"
    "|---|---|---|---|---|\n"
    "| 2026-09-17 | Se usan 50 árboles en Random Forest | Se obtiene un baseline estable con tiempo de cómputo manejable | Gabriel Elizondo | Define la configuración inicial del modelo |\n"
    "| 2026-09-17 | Se fija la semilla 42 | Permite repetir la generación de datos, la partición y el entrenamiento | Gabriel Elizondo | Facilita la comparación entre ejecuciones |\n"
)

Path("DECISION_LOG.md").write_text(decision_log, encoding="utf-8")
print(Path("DECISION_LOG.md").read_text(encoding="utf-8"))


| Fecha | Decisión | Evidencia | Responsable | Impacto |
|---|---|---|---|---|
| 2026-09-17 | Se usan 50 árboles en Random Forest | Se obtiene un baseline estable con tiempo de cómputo manejable | Gabriel Elizondo | Define la configuración inicial del modelo |
| 2026-09-17 | Se fija la semilla 42 | Permite repetir la generación de datos, la partición y el entrenamiento | Gabriel Elizondo | Facilita la comparación entre ejecuciones |



Registré dos decisiones que afectan directamente la trazabilidad del experimento. La bitácora deja evidencia de qué se decidió, por qué se hizo y qué impacto tiene, sin depender de la memoria de quien ejecutó el proyecto.


## Reto 4. Matriz RACI para promoción del modelo


In [ ]:
import pandas as pd

# assign governance roles for the promotion decision
raci = pd.DataFrame([
    {
        "Actividad": "Promover la versión a producción",
        "Propietario del modelo": "A - aprueba la promoción",
        "Especialista técnico": "R - ejecuta y prepara la evidencia técnica",
        "Cumplimiento / ética": "C / I - revisa riesgos y queda informado del resultado",
    }
])

print(raci.to_string(index=False))
print("\nEl aprobador no debe ser la misma persona que ejecuta el código porque separa la revisión de la ejecución y reduce conflictos de interés.")


                       Actividad   Propietario del modelo                       Especialista técnico                                   Cumplimiento / ética
Promover la versión a producción A - aprueba la promoción R - ejecuta y prepara la evidencia técnica C / I - revisa riesgos y queda informado del resultado

El aprobador no debe ser la misma persona que ejecuta el código porque separa la revisión de la ejecución y reduce conflictos de interés.


Asigné al propietario como aprobador final, al especialista técnico como responsable de ejecutar la promoción y a cumplimiento/ética como consultado e informado. Separar la aprobación de la ejecución permite que la promoción tenga una revisión independiente.


## Conclusión

En esta práctica comprobé que un buen F1-score no es suficiente para promover un modelo. También necesito poder reproducirlo, justificar las decisiones tomadas y definir quién ejecuta, consulta y aprueba cada cambio.
